# NB_00_SOURCE_EXTRACTION

This notebook converts one engineering source into structured, reviewable Reading Point inputs.

**Source workflow**

```text
Engineering source
        ↓
Measured engineering states
        ↓
Target specifications
        ↓
Engineering constraints
        ↓
Engineering refinements
        ↓
Candidate Reading Point dialogue
```

This v1 is grounded in Dan Becker's presentation:

> *Achieving 1% Assay of Special Nuclear Materials in 2 Minutes with Microcalorimeter-Array Gamma-Ray Spectroscopy*  
> ARPA-E Fission Annual Meeting, October 1–2, 2025.

The notebook does not infer missing measurements. Every extracted item should remain traceable to a source page.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import json
import zipfile

try:
    import yaml
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pyyaml"],
        check=True,
    )
    import yaml

try:
    import pandas as pd
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pandas"],
        check=True,
    )
    import pandas as pd

from IPython.display import Markdown, display

NOTEBOOK_ID = "NB_00_SOURCE_EXTRACTION"
NOTEBOOK_VERSION = "1.0.0"
REPOSITORY = "sensors-becker"

OUTPUT_DIRECTORY = Path("outputs/source_extraction/becker_2025_arpa_e")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": REPOSITORY,
    "output_directory": str(OUTPUT_DIRECTORY),
}


## Source Identity

The source record establishes provenance before any engineering statement is written.


In [ ]:
SOURCE = {
    "source_id": "BECKER_2025_ARPA_E_MICROCALORIMETER_ASSAY",
    "title": (
        "Achieving 1% Assay of Special Nuclear Materials in 2 Minutes "
        "with Microcalorimeter-Array Gamma-Ray Spectroscopy"
    ),
    "author": "Dan Becker",
    "organization": "University of Colorado",
    "event": "ARPA-E Fission Annual Meeting",
    "date": "2025-10-01/2025-10-02",
    "source_file": "Daniel Becker (1).pdf",
    "page_count": 16,
    "engineering_object": "Microcalorimeter",
    "engineering_direction": "Toward next-generation microcalorimeters.",
}

SOURCE


## Extraction Schema

Each item records:

- the source page;
- the source-derived engineering statement;
- the engineering category;
- any reported value, unit, and comparison state;
- a short note explaining how the item is used.

The categories in this notebook are:

```text
measured_state
target_specification
engineering_constraint
engineering_refinement
deployment_requirement
```


In [ ]:
@dataclass(frozen=True)
class SourceExtraction:
    extraction_id: str
    category: str
    statement: str
    page: int
    metric: str | None = None
    value: float | int | str | None = None
    unit: str | None = None
    comparison_state: str | None = None
    note: str = ""

    def validate(self) -> None:
        allowed = {
            "measured_state",
            "target_specification",
            "engineering_constraint",
            "engineering_refinement",
            "deployment_requirement",
        }
        if self.category not in allowed:
            raise ValueError(f"Unsupported category: {self.category}")
        if self.page < 1 or self.page > SOURCE["page_count"]:
            raise ValueError(f"Invalid source page: {self.page}")
        if not self.statement.strip():
            raise ValueError("statement is required")


## Source-Derived Extractions

The following entries are derived directly from the presentation. Edit or extend this cell as additional sources are reviewed.


In [ ]:
EXTRACTIONS = [
    SourceExtraction(
        extraction_id="MS_001",
        category="measured_state",
        statement="Current CURIE detector speed is compatible with 140 photons per second.",
        page=12,
        metric="per_detector_count_rate",
        value=140,
        unit="counts_per_second",
        comparison_state="current",
        note="Reported as within 1.4× of the 200 cps target.",
    ),
    SourceExtraction(
        extraction_id="MS_002",
        category="measured_state",
        statement="Current pulse fall time to 1% of peak is approximately 1.5 milliseconds.",
        page=16,
        metric="fall_time_to_1_percent",
        value=1.5,
        unit="milliseconds",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="MS_003",
        category="measured_state",
        statement="Current athermal tails are negligible.",
        page=16,
        metric="athermal_tails",
        value="negligible",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="MS_004",
        category="measured_state",
        statement="Current energy resolution is approximately 150 electronvolts at 100 kiloelectronvolts.",
        page=16,
        metric="energy_resolution_at_100_keV",
        value=150,
        unit="electronvolts",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="MS_005",
        category="measured_state",
        statement="The all-silicon detector architecture has produced approximately a 14× speed improvement.",
        page=16,
        metric="speed_improvement",
        value=14,
        unit="times",
        comparison_state="current",
    ),
    SourceExtraction(
        extraction_id="TS_001",
        category="target_specification",
        statement="Increase per-detector count rate to 200 counts per second.",
        page=7,
        metric="per_detector_count_rate",
        value=200,
        unit="counts_per_second",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="TS_002",
        category="target_specification",
        statement="Maintain energy resolution below 100 electronvolts at 100 kiloelectronvolts.",
        page=7,
        metric="energy_resolution_at_100_keV",
        value="<100",
        unit="electronvolts",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="TS_003",
        category="target_specification",
        statement="Reduce pulse fall time to 750 microseconds.",
        page=7,
        metric="fall_time_to_1_percent",
        value=750,
        unit="microseconds",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="TS_004",
        category="target_specification",
        statement="Enable assay of complex special nuclear material mixtures to within 1% accuracy in 2 minutes.",
        page=2,
        metric="assay_accuracy_and_time",
        value="1% in 2 minutes",
        comparison_state="target",
    ),
    SourceExtraction(
        extraction_id="EC_001",
        category="engineering_constraint",
        statement="Detector decay time limits count rate.",
        page=3,
        metric="decay_time",
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="EC_002",
        category="engineering_constraint",
        statement="The pre-CURIE membrane architecture is finicky to assemble, fragile, and limited in its ability to increase thermal conductance G.",
        page=9,
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="EC_003",
        category="engineering_constraint",
        statement="Energy resolution degraded as detector speed increased.",
        page=12,
        metric="energy_resolution_at_100_keV",
        comparison_state="constraint",
    ),
    SourceExtraction(
        extraction_id="ER_001",
        category="engineering_refinement",
        statement="Use an all-silicon, membrane-free detector architecture.",
        page=10,
        comparison_state="refinement",
        note="Presented as forgiving to assemble, physically robust, and providing access to a wide range of G.",
    ),
    SourceExtraction(
        extraction_id="ER_002",
        category="engineering_refinement",
        statement="Lower the operating temperature and superconducting critical temperature Tc.",
        page=14,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="ER_003",
        category="engineering_refinement",
        statement="Increase coupling to silicon.",
        page=14,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="ER_004",
        category="engineering_refinement",
        statement="Solve the absorber manufacturing problem.",
        page=15,
        comparison_state="refinement",
    ),
    SourceExtraction(
        extraction_id="DR_001",
        category="deployment_requirement",
        statement="Validate detector performance with relevant samples from INL's Hot Fuel Examination Facility.",
        page=7,
        comparison_state="deployment",
    ),
    SourceExtraction(
        extraction_id="DR_002",
        category="deployment_requirement",
        statement="Define requirements for product assembly.",
        page=15,
        comparison_state="deployment",
    ),
]

for item in EXTRACTIONS:
    item.validate()

len(EXTRACTIONS)


## Review Extracted Engineering Content

The table is the primary human-review checkpoint. It should be inspected before any Reading Point YAML is generated.


In [ ]:
extraction_table = pd.DataFrame(asdict(item) for item in EXTRACTIONS)
extraction_table[
    [
        "extraction_id",
        "category",
        "page",
        "metric",
        "value",
        "unit",
        "statement",
    ]
]


## Current-to-Target Comparison

Only metrics with explicit current and target states are compared here.


In [ ]:
comparison_rows = [
    {
        "metric": "Per-detector count rate",
        "current": "140 cps",
        "target": "200 cps",
        "source_pages": "12, 16 / 7",
    },
    {
        "metric": "Energy resolution at 100 keV",
        "current": "150 eV",
        "target": "<100 eV",
        "source_pages": "16 / 7",
    },
    {
        "metric": "Fall time to 1% of peak",
        "current": "~1.5 ms",
        "target": "750 µs",
        "source_pages": "16 / 7",
    },
    {
        "metric": "Athermal tails",
        "current": "Negligible",
        "target": "Negligible",
        "source_pages": "16 / 7",
    },
]

comparison_table = pd.DataFrame(comparison_rows)
comparison_table


## Candidate Source-Derived Reading Point

The Reading Point grammar remains in metadata. The visible dialogue carries source-specific engineering content.

```text
Current detector performance
        ↓
Target detector performance
        ↓
Priority detector refinements
        ↓
Engineering sessions
```


In [ ]:
READING_POINT_CANDIDATE = {
    "reading_point_id": "RP_37",
    "engineering_object": "Microcalorimeter",
    "source_id": SOURCE["source_id"],
    "grammar": {
        "A": "Measured Engineering Improvement informs Leading Specifications.",
        "B": "Leading Specifications direct Engineering Priorities.",
        "C": "Engineering Priorities prepare Engineering Sessions.",
    },
    "dialogue": [
        {
            "order": "A",
            "concept": "Leading Specifications",
            "title": "Detector Performance Targets: Microcalorimeters",
            "first_label": "140 cps · 150 eV · ~1.5 ms",
            "second_label": "200 cps · <100 eV · 750 µs",
            "supporting_context": [
                "Negligible Athermal Tails",
                "1% Assay in 2 Minutes",
            ],
            "engineering_statement": (
                "Measured detector performance specifies the next detector-performance targets."
            ),
        },
        {
            "order": "B",
            "concept": "Engineering Priorities",
            "title": "Detector Refinement Priorities: Microcalorimeters",
            "first_label": "200 cps · <100 eV · 750 µs",
            "second_label": "Lower Tc · Increase Si Coupling",
            "supporting_context": [
                "All-Silicon Architecture",
                "Absorber Manufacturing",
            ],
            "engineering_statement": (
                "Target detector performance directs detector-refinement priorities."
            ),
        },
        {
            "order": "C",
            "concept": "Engineering Sessions",
            "title": "Engineering Sessions: Microcalorimeters",
            "first_label": "Lower Tc · Increase Si Coupling",
            "second_label": "Fabricate · Characterize · Validate",
            "supporting_context": [
                "Relevant INL Samples",
                "Product Assembly Requirements",
            ],
            "engineering_statement": (
                "Detector-refinement priorities prepare fabrication, characterization, "
                "and validation sessions."
            ),
        },
    ],
}

READING_POINT_CANDIDATE


## Export Source Record

The notebook exports:

- the complete source extraction as YAML and JSON;
- a Markdown review record;
- a candidate RP_37 YAML;
- a ZIP bundle containing all generated records.


In [ ]:
source_record = {
    "source": SOURCE,
    "extractions": [asdict(item) for item in EXTRACTIONS],
    "current_to_target": comparison_rows,
    "reading_point_candidate": READING_POINT_CANDIDATE,
}

json_path = OUTPUT_DIRECTORY / "becker_2025_source_extraction.json"
yaml_path = OUTPUT_DIRECTORY / "becker_2025_source_extraction.yaml"
review_path = OUTPUT_DIRECTORY / "becker_2025_source_extraction.md"
rp_path = OUTPUT_DIRECTORY / "RP_37_SOURCE_DERIVED.yaml"
zip_path = OUTPUT_DIRECTORY / "NB_00_SOURCE_EXTRACTION.zip"

json_path.write_text(
    json.dumps(source_record, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
yaml_path.write_text(
    yaml.safe_dump(source_record, sort_keys=False, allow_unicode=True, width=100),
    encoding="utf-8",
)
rp_path.write_text(
    yaml.safe_dump(
        READING_POINT_CANDIDATE,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    ),
    encoding="utf-8",
)

review_lines = [
    f"# Source Extraction: {SOURCE['author']}",
    "",
    f"**Source:** {SOURCE['title']}",
    "",
    "## Current-to-Target Comparison",
    "",
    comparison_table.to_markdown(index=False),
    "",
    "## Engineering Constraints",
    "",
]
for item in EXTRACTIONS:
    if item.category == "engineering_constraint":
        review_lines.append(f"- Page {item.page}: {item.statement}")

review_lines.extend(["", "## Engineering Refinements", ""])
for item in EXTRACTIONS:
    if item.category == "engineering_refinement":
        review_lines.append(f"- Page {item.page}: {item.statement}")

review_lines.extend(
    [
        "",
        "## Candidate Reading Point",
        "",
        "```text",
        "140 cps · 150 eV · ~1.5 ms",
        "        ↓",
        "200 cps · <100 eV · 750 µs",
        "        ↓",
        "Lower Tc · Increase Si Coupling",
        "        ↓",
        "Fabricate · Characterize · Validate",
        "```",
        "",
        "*Admissible generalizations trail leading specifications.*",
    ]
)

review_path.write_text("\n".join(review_lines) + "\n", encoding="utf-8")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (json_path, yaml_path, review_path, rp_path):
        archive.write(path, arcname=path.name)

generated = {
    "json": json_path,
    "yaml": yaml_path,
    "review": review_path,
    "reading_point_candidate": rp_path,
    "zip": zip_path,
}

generated


## Verification

This final cell verifies that every generated artifact exists and contains data.


In [ ]:
for label, path in generated.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    if path.stat().st_size <= 0:
        raise ValueError(f"Empty {label}: {path}")

print("Source extraction bundle: VERIFIED")
for label, path in generated.items():
    print(f"{label}: {path} ({path.stat().st_size} bytes)")
